# Анализ эксперимента

Срок вышел, эксперимент остановлен. Путь: sanity-checks → целевая метрика → прокси и защитные → решение. Результаты каждого шага прикладывай в карточку.

Данные: события в ClickHouse (`ab.events`), абшница — реплика в ClickHouse (`ab.assignments`), конфиг эксперимента — в Postgres (`ab.experiments`).

In [ ]:
# pip install clickhouse-connect psycopg2-binary pandas numpy scipy statsmodels
import numpy as np
import pandas as pd
from clickhouse_connect.dbapi import connect as ch_connect
import psycopg2

EXPERIMENT_CODE = "pricing_point_estimate"   # код твоего эксперимента

ch_cursor = ch_connect(host="localhost", port=8123, username="default", password="platform").cursor()
pg_conn = psycopg2.connect(host="localhost", port=5434, dbname="platform", user="platform", password="platform")

def q_ch(sql: str) -> pd.DataFrame:
    ch_cursor.execute(sql)
    return pd.DataFrame(ch_cursor.fetchall(), columns=[c[0] for c in ch_cursor.description])

def q_pg(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, pg_conn)

# Конфиг эксперимента: id, даты старта и остановки, варианты
exp = q_pg(f"SELECT * FROM ab.experiments WHERE code = '{EXPERIMENT_CODE}'")
exp

## Шаг 1. Sanity-checks

Проверки идут до метрик: если сломан сплит или логирование, метрики не имеют смысла.

1. **SRM**: финальные размеры групп против плановых долей — хи-квадрат.
2. **Набор по дням**: без аномальных провалов и всплесков.
3. **Полнота логов**: события есть за каждый день эксперимента.

In [ ]:
from scipy import stats

# TODO: размеры групп из ab.assignments (ClickHouse, не забудь FINAL)

# TODO: хи-квадрат против плановых долей: stats.chisquare(observed, expected)

# TODO: назначения по дням (assigned_at::date) и события по дням — глазами по таблице или графиком

## Шаг 2. Целевая метрика

Метрика-доля → z-тест двух долей. Считай на пользователя: попал в эксперимент (есть в абшнице) / сконвертировался (есть `order_confirm` после `assigned_at`).

Нужны три числа на группу и три результата: p-value, доверительный интервал разницы, сравнение эффекта с MDE из карточки.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

# TODO: SQL — по каждой группе: пользователей всего, пользователей с конверсией
#   (join events с ab.assignments FINAL, событие после assigned_at)

# TODO: z-тест: proportions_ztest([успехи_A, успехи_B], [n_A, n_B])

# TODO: доверительный интервал разницы долей: confint_proportions_2indep(...)

# TODO: вывод — эффект в п.п., его интервал, отношение к MDE из карточки

## Шаг 3. Прокси и защитные метрики

Прокси показывают механизм эффекта по воронке. Защитные — цену эффекта.

У `cancel_rate` юнит анализа (заказ) мельче юнита рандомизации (пользователь): заказы одного пользователя зависимы, наивный z-тест по заказам занижает дисперсию и даёт ложную значимость. Корректные способы — дельта-метод или бутстрап **по пользователям**: сэмплируются пользователи со всеми их заказами.

In [ ]:
# TODO: по каждому пользователю — заказы и отмены (SQL, группировка по user_id)

def bootstrap_ratio_diff(df_a: pd.DataFrame, df_b: pd.DataFrame,
                         n_iter: int = 5000, seed: int = 42) -> np.ndarray:
    """Разница cancel_rate (B - A) по бутстрап-выборкам пользователей."""
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_iter):
        # TODO: сэмплируй пользователей с возвращением в каждой группе
        # TODO: ratio группы = sum(отмен) / sum(заказов) по сэмплу
        ...
    return np.array(diffs)

# TODO: доверительный интервал = квантили diffs (2.5%, 97.5%); значимость — содержит ли интервал ноль

## Шаг 4. Выводы и решение

Заполни и перенеси в карточку:

- эффект по целевой метрике: размер, интервал, значимость, отношение к MDE;
- что показали прокси: сработал ли заявленный механизм;
- что показали защитные: цена эффекта;
- решение: раскатываем / не раскатываем / дорабатываем — с аргументами;
- ограничения и риски: длительность, сезонность, качество данных.

In [ ]:
# Твои выводы